In [1]:
!pip install rank_bm25 openai numpy

In [2]:
import os
from getpass import getpass
os.environ['OPENAI_API_KEY'] = getpass('Enter OpenAI API Key: ')

Enter OpenAI API Key: ··········


In [3]:
import math
import re
import numpy as np
from collections import Counter
from rank_bm25 import BM25Okapi
from openai import OpenAI

In [4]:
client = OpenAI()

# Defining the Corpus
Before comparing BM25 and vector search, we need a shared knowledge base to search over. We define 12 short text chunks covering a range of topics — Python, machine learning, BM25, transformers, embeddings, RAG, databases, and more. The topics are deliberately varied: some chunks are closely related (BM25 and TF-IDF, embeddings and cosine similarity), while others are completely unrelated (PostgreSQL, Django). This variety is what makes the comparison meaningful — a retrieval method that works well should surface the relevant chunks and ignore the noise.

This corpus acts as our stand-in for a real document store. In a production RAG pipeline, these chunks would come from splitting and cleaning actual documents — PDFs, wikis, knowledge bases. Here, we keep them short and hand-crafted so the retrieval behaviour is easy to trace and reason about.

In [5]:
CHUNKS = [
    # 0
    "Python is a high-level, interpreted programming language known for its simple and readable syntax. "
    "It supports multiple programming paradigms including procedural, object-oriented, and functional programming.",

    # 1
    "Machine learning is a subset of artificial intelligence that enables systems to learn from data "
    "without being explicitly programmed. Common algorithms include linear regression, decision trees, and neural networks.",

    # 2
    "BM25 stands for Best Match 25. It is a bag-of-words retrieval function used by search engines "
    "to rank documents based on the query terms appearing in each document. "
    "BM25 uses term frequency and inverse document frequency with length normalization.",

    # 3
    "Transformer architecture introduced the self-attention mechanism, which allows the model to weigh "
    "the importance of different words in a sentence regardless of their position. "
    "BERT and GPT are both based on the Transformer architecture.",

    # 4
    "Vector embeddings represent text as dense numerical vectors in a high-dimensional space. "
    "Similar texts are placed closer together. This allows semantic search — finding documents "
    "that mean the same thing even if they use different words.",

    # 5
    "TF-IDF stands for Term Frequency–Inverse Document Frequency. It reflects how important a word is "
    "to a document relative to the entire corpus. Rare words get higher scores than common ones like 'the'.",

    # 6
    "Retrieval-Augmented Generation (RAG) combines a retrieval system with a language model. "
    "The retriever finds relevant documents; the generator uses them as context to produce an answer. "
    "This reduces hallucinations and allows the model to cite sources.",

    # 7
    "Django is a high-level Python web framework that encourages rapid development and clean, pragmatic design. "
    "It includes an ORM, authentication system, and admin panel out of the box.",

    # 8
    "Cosine similarity measures the angle between two vectors. A score of 1 means identical direction, "
    "0 means orthogonal, and -1 means opposite. It is commonly used to compare text embeddings.",

    # 9
    "Gradient descent is an optimization algorithm used to minimize a loss function by iteratively "
    "moving in the direction of the steepest descent. It is the backbone of training neural networks.",

    # 10
    "PostgreSQL is an open-source relational database known for its robustness and support for advanced "
    "SQL features like window functions, CTEs, and JSON storage.",

    # 11
    "Sparse retrieval methods like BM25 rely on exact keyword matches and fail when the query uses "
    "synonyms or paraphrases not present in the document. Dense retrieval using embeddings handles "
    "this by matching semantic meaning rather than surface form.",
]

print(f"Corpus loaded: {len(CHUNKS)} chunks")
for i, c in enumerate(CHUNKS):
    print(f"  [{i:02d}] {c[:75]}...")


Corpus loaded: 12 chunks
  [00] Python is a high-level, interpreted programming language known for its simp...
  [01] Machine learning is a subset of artificial intelligence that enables system...
  [02] BM25 stands for Best Match 25. It is a bag-of-words retrieval function used...
  [03] Transformer architecture introduced the self-attention mechanism, which all...
  [04] Vector embeddings represent text as dense numerical vectors in a high-dimen...
  [05] TF-IDF stands for Term Frequency–Inverse Document Frequency. It reflects ho...
  [06] Retrieval-Augmented Generation (RAG) combines a retrieval system with a lan...
  [07] Django is a high-level Python web framework that encourages rapid developme...
  [08] Cosine similarity measures the angle between two vectors. A score of 1 mean...
  [09] Gradient descent is an optimization algorithm used to minimize a loss funct...
  [10] PostgreSQL is an open-source relational database known for its robustness a...
  [11] Sparse retrieval metho

# Building the BM25 Retriever
With the corpus defined, we can build the BM25 index. The process has two steps: tokenization and indexing. The tokenize function lowercases the text and splits on any non-alphanumeric character — so "TF-IDF" becomes ["tf", "idf"] and "bag-of-words" becomes ["bag", "of", "words"]. This is intentionally simple: BM25 is a bag-of-words model, so there is no stemming, no stopword removal, and no linguistic preprocessing. Every word is treated as an independent token.

Once every chunk is tokenized, BM25Okapi builds the index — computing document lengths, average document length, and IDF scores for every unique term in the corpus. This happens once at startup. At query time, bm25_search tokenizes the incoming query the same way, calls get_scores to compute a BM25 relevance score for every chunk in parallel, then sorts and returns the top-k results. The sanity check at the bottom runs a test query to confirm the index is working before we move on to the embedding retriever.

In [6]:
def tokenize(text: str) -> list[str]:
    """Lowercase and split on non-alphanumeric characters."""
    return re.findall(r'\w+', text.lower())

# Build BM25 index over the corpus
tokenized_corpus = [tokenize(chunk) for chunk in CHUNKS]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_search(query: str, top_k: int = 3) -> list[dict]:
    """Return top-k chunks ranked by BM25 score."""
    tokens = tokenize(query)
    scores = bm25.get_scores(tokens)
    ranked = np.argsort(scores)[::-1][:top_k]
    return [
        {"chunk_id": int(i), "score": round(float(scores[i]), 4), "text": CHUNKS[i]}
        for i in ranked
    ]

# Quick sanity check
results = bm25_search("how does BM25 rank documents", top_k=3)
print("BM25 test — query: 'how does BM25 rank documents'")
for r in results:
    print(f"  [{r['chunk_id']}] score={r['score']}  {r['text'][:70]}...")

BM25 test — query: 'how does BM25 rank documents'
  [2] score=4.6069  BM25 stands for Best Match 25. It is a bag-of-words retrieval function...
  [5] score=1.9954  TF-IDF stands for Term Frequency–Inverse Document Frequency. It reflec...
  [11] score=1.3335  Sparse retrieval methods like BM25 rely on exact keyword matches and f...


# Building the Embedding Retriever
The embedding retriever works differently from BM25 at every step. Instead of counting tokens, it converts each chunk into a dense numerical vector — a list of 1,536 numbers — using OpenAI's text-embedding-3-small model. Each number represents a dimension in semantic space, and chunks that mean similar things end up with vectors that point in similar directions, regardless of the words they use.

The index build step calls the embedding API once per chunk and stores the resulting vectors in memory. This is the key cost difference from BM25: building the BM25 index is pure arithmetic on your own machine, while building the embedding index requires one API call per chunk and produces vectors you need to store. For 12 chunks this is trivial; at a million chunks, this becomes a real infrastructure decision.

At query time, embedding_search embeds the incoming query using the same model — this is important, the query and the chunks must live in the same vector space — then computes cosine similarity between the query vector and every stored chunk vector. Cosine similarity measures the angle between two vectors: a score of 1 means identical direction, 0 means completely unrelated, and negative values mean opposite meaning. The chunks are then ranked by this score and the top-k are returned. The same sanity check query from the BM25 section runs here too, so you can see the first direct comparison between the two approaches on identical input.

In [7]:
EMBED_MODEL = "text-embedding-3-small"

def get_embedding(text: str) -> np.ndarray:
    response = client.embeddings.create(model=EMBED_MODEL, input=text)
    return np.array(response.data[0].embedding)

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Embed all chunks once (this is the "index build" step in RAG)
print("Building embedding index... (one API call per chunk)")
chunk_embeddings = [get_embedding(chunk) for chunk in CHUNKS]
print(f"Done. Each embedding has {len(chunk_embeddings[0])} dimensions.")

def embedding_search(query: str, top_k: int = 3) -> list[dict]:
    """Return top-k chunks ranked by cosine similarity to the query embedding."""
    query_emb = get_embedding(query)
    scores = [cosine_similarity(query_emb, emb) for emb in chunk_embeddings]
    ranked = np.argsort(scores)[::-1][:top_k]
    return [
        {"chunk_id": int(i), "score": round(float(scores[i]), 4), "text": CHUNKS[i]}
        for i in ranked
    ]
# Quick sanity check
results = embedding_search("how does BM25 rank documents", top_k=3)
print("\nEmbedding test — query: 'how does BM25 rank documents'")
for r in results:
    print(f"  [{r['chunk_id']}] score={r['score']}  {r['text'][:70]}...")

Building embedding index... (one API call per chunk)
Done. Each embedding has 1536 dimensions.

Embedding test — query: 'how does BM25 rank documents'
  [2] score=0.7092  BM25 stands for Best Match 25. It is a bag-of-words retrieval function...
  [11] score=0.4406  Sparse retrieval methods like BM25 rely on exact keyword matches and f...
  [5] score=0.2812  TF-IDF stands for Term Frequency–Inverse Document Frequency. It reflec...


# Side-by-Side Comparison Function
This is the core of the experiment. The compare function runs the same query through both retrievers simultaneously and prints the results in a two-column layout — BM25 on the left, embeddings on the right — so the differences are immediately visible at the same rank position.

In [8]:
def compare(query: str, top_k: int = 3):
    bm25_results    = bm25_search(query, top_k)
    embed_results   = embedding_search(query, top_k)

    print(f"\n{'═'*70}")
    print(f"  QUERY: \"{query}\"")
    print(f"{'═'*70}")

    print(f"\n  {'BM25 (keyword)':<35}  {'Embedding RAG (semantic)'}")
    print(f"  {'─'*33}  {'─'*33}")

    for rank, (b, e) in enumerate(zip(bm25_results, embed_results), 1):
        b_preview = b['text'][:55].replace('\n', ' ')
        e_preview = e['text'][:55].replace('\n', ' ')
        same = "⬅ same" if b['chunk_id'] == e['chunk_id'] else ""
        print(f"  #{rank} [{b['chunk_id']:02d}] {b['score']:.4f}  {b_preview}...")
        print(f"     [{e['chunk_id']:02d}] {e['score']:.4f}  {e_preview}...  {same}")
        print()

In [9]:
compare("BM25 term frequency inverse document frequency")


══════════════════════════════════════════════════════════════════════
  QUERY: "BM25 term frequency inverse document frequency"
══════════════════════════════════════════════════════════════════════

  BM25 (keyword)                       Embedding RAG (semantic)
  ─────────────────────────────────  ─────────────────────────────────
  #1 [02] 9.5572  BM25 stands for Best Match 25. It is a bag-of-words ret...
     [02] 0.6900  BM25 stands for Best Match 25. It is a bag-of-words ret...  ⬅ same

  #2 [05] 8.2577  TF-IDF stands for Term Frequency–Inverse Document Frequ...
     [05] 0.5599  TF-IDF stands for Term Frequency–Inverse Document Frequ...  ⬅ same

  #3 [11] 2.2614  Sparse retrieval methods like BM25 rely on exact keywor...
     [11] 0.4352  Sparse retrieval methods like BM25 rely on exact keywor...  ⬅ same



This is the cleanest result in the entire experiment — both methods agree on all three chunks, in the same order. The query contains precise, rare, domain-specific terms ("BM25", "term frequency", "inverse document frequency") that appear verbatim in exactly the right documents. BM25 scores are high and well-separated (9.55, 8.25, 2.26), signalling strong confidence. Embedding scores are equally healthy (0.69, 0.55, 0.43), meaning the semantic space also places these chunks close to the query vector.

In [11]:
compare("what is RAG and why does it reduce hallucinations")


══════════════════════════════════════════════════════════════════════
  QUERY: "what is RAG and why does it reduce hallucinations"
══════════════════════════════════════════════════════════════════════

  BM25 (keyword)                       Embedding RAG (semantic)
  ─────────────────────────────────  ─────────────────────────────────
  #1 [06] 4.2593  Retrieval-Augmented Generation (RAG) combines a retriev...
     [06] 0.5278  Retrieval-Augmented Generation (RAG) combines a retriev...  ⬅ same

  #2 [10] 1.2143  PostgreSQL is an open-source relational database known ...
     [09] 0.1478  Gradient descent is an optimization algorithm used to m...  

  #3 [00] 1.1693  Python is a high-level, interpreted programming languag...
     [03] 0.0977  Transformer architecture introduced the self-attention ...  



Both methods correctly return the RAG chunk at #1. After that, BM25 falls apart — it returns PostgreSQL and Python with scores of 1.21 and 1.16, driven purely by stop-word overlap from "is", "and", "why" in the query. The scores look meaningful but are pure noise. Embeddings, by contrast, return low scores (0.15, 0.10) that honestly signal "nothing relevant found." This is the calibration gap: BM25 scores are not comparable across queries, while embedding scores behave like a confidence measure you can actually threshold on.

In [13]:
compare("cosine similarity between vectors")


══════════════════════════════════════════════════════════════════════
  QUERY: "cosine similarity between vectors"
══════════════════════════════════════════════════════════════════════

  BM25 (keyword)                       Embedding RAG (semantic)
  ─────────────────────────────────  ─────────────────────────────────
  #1 [08] 7.8163  Cosine similarity measures the angle between two vector...
     [08] 0.6963  Cosine similarity measures the angle between two vector...  ⬅ same

  #2 [04] 1.3688  Vector embeddings represent text as dense numerical vec...
     [04] 0.4267  Vector embeddings represent text as dense numerical vec...  ⬅ same

  #3 [10] 0.0000  PostgreSQL is an open-source relational database known ...
     [11] 0.2315  Sparse retrieval methods like BM25 rely on exact keywor...  



Both methods agree on #1 and #2. The split happens at #3 — BM25 returns PostgreSQL with a score of 0.0000, meaning it ran out of keyword matches and handed back a random chunk. That zero is not a threshold; it is a fallback. Embeddings return chunk [11] about sparse vs dense retrieval with a score of 0.23 — a genuinely related result, since the chunk discusses how dense retrieval handles cases where keyword matching fails. When keyword matches are exhausted, BM25 returns noise silently while embeddings continue finding meaningful results.